# Quantum random number generator

The difference between Pseudo random numbers and True random numbers.

Computers use pseudo random numbers all the time, For cryptograhpy to randomness in games. Most languages use a number generator from 1997 called `Mersenne Twister`. The algorithm aims to solve several problems with statefull machines like computers. 

## What is randomness?

In most cases true randomness is not needed. For something to feel random to a person it needs to have a lack of patterns, lack of repetetiveness and statistically uniform. Mersenne Twister is named after the french person Marin Mersenne because it follows the period of a Mersenne prime `2^19937-1` along side a few other tricks like the twister and tempering it means that it has a incredibly long sequence, but invertable. While the long period makes it practically unpredictable, it is not unpredictable as one can predict the output of `Mersenne Twists` using the state gotten from previous outputs making a sequence predictable. Given the state values of the `Mersenne Twist` are directly mapped to outputs the state can be recomputed and used to predict following numbers. By seeding the state machine you can set the initials of the machine create a sequence of numbers using the machine. The `temper` operation which manipulates the lower weak bits into a more uniform series by shifting and `XOR`'ing the shifted bits into the original value leaking the higher order bits which are statistically better distributed into the weaker low bits.

## Why this is Pseudo random

Since the exact operation of the temper is known and specified by the creator the output number can be reverted to the original state of the algorithm. This causes an attack surface for reverse engineering the state given an output. Even cryptographically secure random number generator's (CSRNG) can be recomputed when given the origin of entropy and enough data harvesting.

## Why quantum?

For most cases CSRNG is more than enough for functionality, however from an auditing and trust standpoint quantum becomes really interresting as quantum by nature is random since the values only appear after measurement and undefined before the moment of collapse.

In [1]:
import random

randomizer_1 = random.Random(42)
randomizer_2 = random.Random(42)

n_bits = 32

series_1 = [randomizer_1.getrandbits(n_bits) for _ in range(10)]
series_2 = [randomizer_2.getrandbits(n_bits) for _ in range(10)]

assert series_1 == series_2

print("The generated sequences are the same") if series_1 == series_2 else print("The generated sequences are not the same")

The generated sequences are the same


The above code is relevant as it provides proof that given similar conditions 2 different randomizer instances can result in the same outputs. In the following cells we reverse the `mersenne twist` temper steps and display how a exausted period can be used to recreate the machine state in a seperate object.

In [ ]:
def temper(X):
    U = 11
    S = 7
    B = 0x9d2c5680
    T = 15
    C = 0xefc60000
    L = 18

    y1 = X ^ (X >> U)
    y2 = y1 ^ ((y1 << S) & B)
    y3 = y2 ^ ((y2 << T) & C)
    Z = y3 ^ (y3 >> L)

    return Z

def untemper(Z):
    U = 11
    S = 7
    T = 15
    L = 18
    C = 0xefc60000
    B = 0x9d2c5680

    smask = (1 << S) - 1
    umask = (1 << U) - 1
    y1 = Z ^ (Z >> L)
    y2 = y1 ^ ((y1 << T) & C)
    y3 = y2 ^ ((y2 << S) & B & (smask << S))
    y4 = y3 ^ ((y3 << S) & B & (smask << (S*2)))
    y5 = y4 ^ ((y4 << S) & B & (smask << (S*3)))
    y6 = y5 ^ ((y5 << S) & B & (smask << (S*4)))
    y7 = y6 ^ ((y6 >> U) & (umask << (U*2)))
    y8 = y7 ^ ((y7 >> U) & (umask << U))
    X  = y8 ^ ((y8 >> U) & umask)
    return X & 0xffff_ffff


By tempering the value inside the state of the current index track we can display that part of the entropy is not random but instead a blending method of weak and strong random bits.

In [3]:
randomizer_3 = random.Random(100)

randomizer_3.getrandbits(32)
state = randomizer_3.getstate()

rand_x = temper(state[1][1])
rand_y = randomizer_3.getrandbits(32)

assert rand_x == rand_y

print("random ints are the same") if rand_x == rand_y else print("rand ints arent the same")

random ints are the same


For the actual attack or recreation of the random state it can simply be done by untempering all the values inside of a 624 length period and appending the current index to the list to conform to the python implementation of `mersenne twist`. This creates a machine state right before a twist operation to generate the next sequence of random numbers in a seperate cloned machine state.

In [ ]:
def attack(randomizer):
    randoms = [untemper(randomizer.getrandbits(32)) for _ in range(624)]
    randoms.append(624)
    state = (3, tuple(randoms), None)
    new_randomizer = random.Random()
    new_randomizer.setstate(state)
    return new_randomizer


randomizer_4 = random.Random(400)
cloned_randomizer = attack(randomizer_4)

rand_x = randomizer_4.getrandbits(32)
rand_y = cloned_randomizer.getrandbits(32)

assert rand_x == rand_y

print("randomizer_4 and cloned_randomizer return same sequences") if rand_x == rand_y else print("randomizer_4 and cloned_randomizer do not return the same sequence")

randomizer_4 and cloned_randomizer return same sequences


Over the next coming years with the high speed developments of compute speeds and the slow introduction of quantum computing even the traditional cryptographically secure random number generators are in danger. Although these algorithms are inreversable the speed and simulation capabilities of modern computers can speed up the search and current state of the entropy source, and with quantum computers being able to process all the input spaces in a single process the classical entropy sources are soon no longer enough to remain unpredictable.

In [54]:
import secrets

print(secrets.randbelow(2**32))

print(int(secrets.randbits(32)))

1997072229
3999978478


The secrets package implements hardware entropy through several sources, linux has kernel entropy pool seeded by hardware entropy like cpu jitter, interrupt timing, disk activity and network packet arrival timing. While more modern CPU's have hardware instructions `RDRAND` on intel/AMD x86 that use resistor thermal noise as entropy pool.

## The current issue of CSRNG sources

While the current CSRNG algorithms use whitening to redistribute the entropy source into equal probabilities it is unable to adjust the size of the entropy pool. Quantum computers also hurt the whitening process of the algorithms as they are often reliant on hash functions or for example AES based csprng methods. This could weaken the generator not by simulating the entropy source as thermal resistor fluctuations are heavily connected to quantum mechanics of heat transfer between protons, but instead severly weaken the mathmatical whitening side responsible for helping a entropy source become equally distributed, allowing for weaknesses to be created.

Along side this the implementations of entropy sources are often kept propriatary this way it is harder to influence or model the entropy source from the outside. This makes the verification, testing and trusting harder, but also a black box for verifying randomness and security.

In september 2006 Debian accidentally broke the entropy source trying to silence a warning causing the source to be reduced to 32767 possible outcomes causing a large amount of users to use the same keys. Quantum being the true random black box it is easy to verify whether it is truely random as it becomes a system monitoring and health test instead of requiring cryptographic experts to perform and calculate proofs.